## Defining a metric space

In [ ]:
import math

class Instance:
    def __init__(self, features, target):
        self.features = features
        self.target = target

    def __str__(self):
        return f"{self.features} -> {self.target}"

    def __repr__(self):
        return f"Instance({self.features}, {self.target})"

training = [
    Instance([2.50, 6.00], (False)),
    Instance([3.75, 8.00], (False)),
    Instance([2.25, 5.50], (False)),
    Instance([3.25, 8.25], (False)),
    Instance([2.75, 7.50], (False)),
    Instance([4.50, 5.00], (False)),
    Instance([3.50, 5.25], (False)),
    Instance([3.00, 3.25], (False)),
    Instance([4.00, 4.00], (False)),
    Instance([4.25, 3.75], (False)),
    Instance([2.00, 2.00], (False)),
    Instance([5.00, 2.50], (False)),
    Instance([8.25, 8.50], (False)),
    Instance([5.75, 8.75], (True)),
    Instance([4.75, 6.25], (True)),
    Instance([5.50, 6.75], (True)),
    Instance([5.25, 9.50], (True)),
    Instance([7.00, 4.25], (True)),
    Instance([7.50, 8.00], (True)),
    Instance([7.25, 5.75], (True))
]

def euclidean(x, y):
    return math.sqrt(sum((a - b) ** 2 for a, b in zip(x, y)))

def manhattan(x, y):
    return sum(abs(a - b) for a, b in zip(x, y))

## Nearest nieghbor implementation

In [33]:
def find_nn(space, metric, query):
    nn_dist = math.inf
    nn = None

    for instance in space:
        distance = metric(instance.features, query.features)

        if distance < nn_dist:
            nn_dist = distance
            nn = instance

    return nn

def predict(space, metric, query):
    nn = find_nn(space, metric, query)
    return nn.target

query = Instance([6.25, 5.75], None)

prediction = predict(training, euclidean, query)

print(f"Nearest neighbor prediction for {query.features}: {prediction}")

nearest neighbor prediction for [6.25, 5.75]: True


## k-Nearest neighbor

In [41]:
def find_k_nearest(space, metric, query, k):
    k_nearest = []
    for instance in space:
        dist = metric(instance.features, query.features)
        # insert in sorted_space, preserving order
        if len(k_nearest) < k:
            k_nearest.append((dist, instance))
        else:
            farthest = max(k_nearest, key=lambda x: x[0])
            if dist < farthest[0]:
                k_nearest.remove(farthest)
                k_nearest.append((dist, instance))
    
    return [instance for dist, instance in k_nearest]

def majority_vote(neighbors):
    # go through the k-NN, find out which target value appears most often, and return that target value
    counts = {}

    for n in neighbors:
        if n.target not in counts:
            counts[n.target] = 0
        
        counts[n.target] += 1
    
    print(counts)
    return max(counts, key=counts.get)

k = 3

neighbors = find_k_nearest(training, euclidean, query, k)

prediction = majority_vote(neighbors)

print(f"k-Nearest neighbor prediction for {query.features}: {prediction}")

{True: 3}
k-Nearest neighbor prediction for [6.25, 5.75]: True


## Weighted k-Nearest neighbor

In [42]:
def find_weighted_k_nearest(space, metric, query, k):
    k_nearest = []
    for instance in space:
        dist = metric(instance.features, query.features)
        weight = float('inf') if dist == 0 else 1/dist
        
        if len(k_nearest) < k:
            k_nearest.append((dist, weight, instance))
        else:
            farthest = max(k_nearest, key=lambda x: x[0])
            if dist < farthest[0]:
                k_nearest.remove(farthest)
                k_nearest.append((dist, weight, instance))

    return [(instance, weight) for dist, weight, instance in k_nearest]
    
def weighted_majority_vote(weighted_neighbors):
    counts = {}

    for instance in weighted_neighbors:
        if instance[0].target not in counts:
            counts[instance[0].target] = 0
        
        counts[instance[0].target] += 1 * instance[1]

    print(counts)
    return max(counts, key=counts.get)

k = 3

weighted_neighbors = find_weighted_k_nearest(training, euclidean, query, k)

prediction = weighted_majority_vote(weighted_neighbors)

print(f"Weighted k-Nearest neighbor prediction for {query.features}: {prediction}")

{True: 2.432455532033676}
Weighted k-Nearest neighbor prediction for [6.25, 5.75]: True
